In [1]:
import itertools
import numpy as np
import plotly.graph_objects as go

In [2]:
grid = np.load('Data/building_case_0.npy')
wind = np.load('Data/mean_case_0.npy')[:, :, :, [2,1,0]]

In [ ]:
# grid shape: (300, 160, 300) -> (5, 5, 5) using block max pooling
target_shape = (5, 5, 5)

# Automatically compute stride from current wind shape: (300, 160, 300, 3)
sx = grid.shape[0] // target_shape[0]  # 60
sy = grid.shape[1] // target_shape[1]  # 32
sz = grid.shape[2] // target_shape[2]  # 60
reshaped = grid.reshape(target_shape[0], sx, target_shape[1], sy, target_shape[2], sz)
grid_5x5x5 = reshaped.max(axis=(1, 3, 5))

grid_5x5x5.shape

(5, 5, 5)

In [ ]:
# Draw voxel/cubic for grid_5x5x5 (cells > 0 are obstacles)
fig = go.Figure()

def add_cube(fig, x, y, z, size=1.0, color="red", opacity=0.8):
    # 8 vertices of one cube
    verts = np.array([
        [x, y, z],
        [x+size, y, z],
        [x+size, y+size, z],
        [x, y+size, z],
        [x, y, z+size],
        [x+size, y, z+size],
        [x+size, y+size, z+size],
        [x, y+size, z+size],
    ])

    # 12 triangles (2 triangles per face)
    I = [0,0,4,4,0,0,1,1,2,2,3,3]
    J = [1,2,5,6,1,5,2,6,3,7,0,4]
    K = [2,3,6,7,5,4,6,5,7,6,4,7]

    fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=I, j=J, k=K,
        color=color, opacity=opacity,
        flatshading=True, showscale=False
    ))

# Get obstacle cells
occ_idx = np.argwhere(grid_5x5x5 > 0)

# Draw each cube
for xi, yi, zi in occ_idx:
    add_cube(fig, xi, yi, zi, size=1.0, color="crimson", opacity=0.85)

fig.update_layout(
    title="3D Occupancy Cubes (5x5x5)",
    scene=dict(
        xaxis=dict(title="X", range=[0, 5], dtick=1),
        yaxis=dict(title="Y", range=[0, 5], dtick=1),
        zaxis=dict(title="Z", range=[0, 5], dtick=1),
        aspectmode="cube"
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [ ]:

# Automatically compute stride from current wind shape
sx = wind.shape[0] // target_shape[0]  # 60
sy = wind.shape[1] // target_shape[1]  # 32
sz = wind.shape[2] // target_shape[2]  # 60

# Check divisibility for safe reshaping
assert wind.shape[0] % target_shape[0] == 0
assert wind.shape[1] % target_shape[1] == 0
assert wind.shape[2] % target_shape[2] == 0

# (300,160,300,3) -> (5,60,5,32,5,60,3)
reshaped = wind.reshape(
    target_shape[0], sx,
    target_shape[1], sy,
    target_shape[2], sz,
    wind.shape[3]
)

# Mean pooling and std per block
wind_5x5x5 = reshaped.mean(axis=(1, 3, 5))
wind_std = reshaped.std(axis=(1, 3, 5))

print("New grid size:", wind_5x5x5.shape)  # (5, 5, 5, 3)
print("Wind vector at first cell (0,0,0):", wind_5x5x5[0, 0, 0])

Kích thước lưới mới: (5, 5, 5, 3)
Vector gió tại ô đầu tiên (0,0,0): [-1.0008188   0.05201079  2.5510726 ]


In [ ]:
# Draw cubic + 3D wind field for all cells (based on current shape)
nx, ny, nz = grid_5x5x5.shape  # currently (10, 10, 10)

# Build a coordinate grid with the correct shape
X, Y, Z = np.meshgrid(np.arange(nx), np.arange(ny), np.arange(nz), indexing="ij")
x = X.ravel()
y = Y.ravel()
z = Z.ravel()

# Wind vectors
u = wind_5x5x5[..., 0].ravel()
v = wind_5x5x5[..., 1].ravel()
w = wind_5x5x5[..., 2].ravel()

# Filter free cells (not obstacles)
mask = (grid_5x5x5.ravel() <= 0)

# Place cones at cell centers
x_c = x + 0.5
y_c = y + 0.5
z_c = z + 0.5

fig = go.Figure()

# 1) Draw obstacle cubes
occ_idx = np.argwhere(grid_5x5x5 > 0)
for xi, yi, zi in occ_idx:
    add_cube(fig, xi, yi, zi, size=1.0, color="crimson", opacity=0.15)

# 2) Draw wind cones in free cells
fig.add_trace(go.Cone(
    x=x_c[mask], y=y_c[mask], z=z_c[mask],
    u=u[mask], v=v[mask], w=w[mask],
    colorscale="Viridis",
    sizemode="absolute",
    sizeref=2.5,
    anchor="cm",
    colorbar=dict(title="|V|")
))

fig.update_layout(
    title="Obstacle cubes + 3D wind field (obstacles filtered)",
    scene=dict(
        xaxis=dict(title="X", range=[0, nx], dtick=1),
        yaxis=dict(title="Y", range=[0, ny], dtick=1),
        zaxis=dict(title="Z", range=[0, nz], dtick=1),
        aspectmode="cube"
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


# Predefine

In [7]:
# v_levels = [0]  # 0 for startpoint, 1 for endpoint, 2 for intermediate point
# --- Sample parameter configuration ---
drone_params = {
    'rho': 1.18,      # kg/m3
    'Cd': 1,
    'Af': 0.25,        # m2
    'm': 4.0  ,          # kg
    'g_acc': np.array([0, 0, -9.81]),
    'A': 0.45,          # m2
    'Pp_hover':65.0  # Watts
}
speed_map = {0: 10.0,1:15.0}
dt = 1
dt_takeoff = 1
dt_landing = 1

In [8]:
Tmax = 35

In [ ]:
free_cells = np.argwhere(grid_5x5x5 <= 0)  # free cells
idx = np.random.choice(len(free_cells), size=2, replace=False)

start_point = free_cells[idx[0]].tolist()
end_point = free_cells[idx[1]].tolist()

print("start_point:", start_point)
print("end_point:", end_point)

start_point: [0, 2, 0]
end_point: [4, 2, 4]


In [10]:
def calculate_drone_power(v_i, w_i, a_i, params):
    """
    Calculate total power consumption Pi according to the formula sequence.
    v_i, w_i, a_i are numpy array vectors with shape (3,)
    """
    # Get constants from params
    rho = params['rho']       # Air density
    Cd = params['Cd']         # Drag coefficient
    Af = params['Af']         # Frontal area
    m = params['m']           # Drone weight
    g_acc = params['g_acc']   # Gravitational acceleration vector [0, 0, -9.81]
    A = params['A']           # Rotor disk area
    Pp_hover = params['Pp_hover'] # Profile power when hovering
    mg_scalar = m * np.linalg.norm(g_acc)

    # (2) Velocity relative to wind
    v_a = v_i - w_i
    v_a_norm = np.linalg.norm(v_a)

    # (3) Aerodynamic drag
    Di = -0.5 * rho * Cd * Af * v_a_norm * v_a

    # (4) Thrust force vector
    # print("a_i:", np.array(a_i))
    Ti_vec = m * np.array(a_i) - m * g_acc - Di

    # (5) Magnitude of thrust force
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Unit direction vector of thrust
    ti_hat = Ti_vec / Ti_mag if Ti_mag != 0 else np.array([0, 0, 1])

    # (7) Velocity component relative to thrust axis (scalar)
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity (scalar)
    # Formula: v_ind = -1/2*vc + sqrt((1/2*vc)^2 + Ti/(2*rho*A))
    term1 = -0.5 * vc_i
    term2 = np.sqrt((0.5 * vc_i)**2 + Ti_mag / (2 * rho * A))
    v_ind = term1 + term2

    # (9) Useful power
    Pu_i = np.dot(Ti_vec, v_a)

    # (10) Induced power
    Pind_i = Ti_mag * v_ind

    # (11) Profile power
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5

    # (12) Total power
    Pi = Pu_i + Pind_i + Pp_i

    return {
        "Pi": Pi,
        "Pu_i": Pu_i,
        "Pind_i": Pind_i,
        "Pp_i": Pp_i,
        "Ti_vec": Ti_vec,
        "Ti_mag": Ti_mag,
    }

In [ ]:
def generate_moore_neighbor_dict(occupied_map, speed_map, wind_field, cell_size=(4.0, 1.5, 4.0)):
    """
    Create a 3D Moore-neighborhood edge dictionary with:
    - occupied_map: obstacle cells (value > 0)
    - wind_field: wind vector at each cell, shape (nx, ny, nz, 3)
    - cell_size: cell size (sx, sy, sz), in meters
    """
    nx, ny, nz = occupied_map.shape
    moore_dict = {}

    sx, sy, sz = cell_size
    cell_size_vec = np.array([sx, sy, sz], dtype=float)

    # 26 Moore-neighborhood directions
    directions = [d for d in itertools.product([-1, 0, 1], repeat=3) if d != (0, 0, 0)]

    for x, y, z in itertools.product(range(nx), range(ny), range(nz)):
        if occupied_map[x, y, z] > 0:
            continue

        for dx, dy, dz in directions:
            x2, y2, z2 = x + dx, y + dy, z + dz

            if not (0 <= x2 < nx and 0 <= y2 < ny and 0 <= z2 < nz):
                continue
            if occupied_map[x2, y2, z2] > 0:
                continue

            # Edge vector in cell units and in meters
            edge_vec_cell = np.array([dx, dy, dz], dtype=float)
            edge_vec_meter = edge_vec_cell * cell_size_vec
            distance = np.linalg.norm(edge_vec_meter)
            if distance == 0:
                continue

            # Average wind at both edge endpoints
            w_i = 0.5 * (wind_field[x, y, z] + wind_field[x2, y2, z2])

            for v_level, speed in speed_map.items():
                # Drone velocity along edge direction (m/s), normalized by true distance
                v_i = (speed / distance) * edge_vec_meter

                power_result = calculate_drone_power(
                    v_i, w_i, np.array([0.0, 0.0, 0.0]), drone_params
                )

                edge_key = f"{x}_{y}_{z}_{x2}_{y2}_{z2}_{v_level}"
                moore_dict[edge_key] = {
                    "startpoint": [x, y, z],
                    "endpoint": [x2, y2, z2],
                    "euclidean_distance": distance,
                    "v_level": v_level,
                    "v": speed,
                    "v_vector": v_i,
                    "wind_vector": w_i,
                    "Pi": power_result["Pi"],
                    "Tmaneuver": power_result["Ti_mag"],
                    "Energy": power_result["Pi"] * (distance / speed),
                }

    return moore_dict

In [12]:
def calculate_energy_transition(edge_in_key, edge_out_key, wind, params,dt):
    """
    Calculate energy at the intersection point between 2 edges based on Vlevel
    """
    # 1. Decode edge information
    # edge_in_key=space_graph[edge_in_key]
    # edge_out_key=space_graph[edge_out_key]
    p_prev, p_curr, v_in_vec,v_mag_in = edge_in_key["startpoint"], edge_in_key["endpoint"], edge_in_key["v_vector"],edge_in_key["v"]
    _, p_next, v_out_vec,v_mag_out = edge_out_key["startpoint"], edge_out_key["endpoint"], edge_out_key["v_vector"],edge_out_key["v"]

  
    # 3. Calculate v_i and a_i at point p_curr
    # Assume dt is the transition time (average distance / average velocity)
    # dist_avg = (np.linalg.norm(p_curr - p_prev) + np.linalg.norm(p_next - p_curr)) / 2
    # dt = dist_avg / ((v_mag_in + v_mag_out) / 2)

    v_i = (v_in_vec + v_out_vec) / 2
    a_i = (v_out_vec - v_in_vec) / dt

    # --- Physical sequence ---
    rho, Cd, Af, m, A = params['rho'], params['Cd'], params['Af'], params['m'], params['A']
    g_vec = params['g_acc']
    Pp_hover = params['Pp_hover']
    mg_scalar = m * np.linalg.norm(g_vec)

    # (2) Relative velocity
    v_a = v_i - wind
    va_norm = np.linalg.norm(v_a)

    # (3) Drag force Di
    Di = -0.5 * rho * Cd * Af * va_norm * v_a

    # (4) Thrust force Ti (Vector)
    Ti_vec = m * a_i - m * g_vec - Di
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Thrust direction
    ti_hat = Ti_vec / Ti_mag if Ti_mag > 1e-6 else np.array([0, 0, 1])

    # (7) Velocity along thrust axis
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity
    v_ind = -0.5 * vc_i + np.sqrt(max(0, (0.5 * vc_i)**2 + Ti_mag / (2 * rho * A)))

    # (9-12) Power components
    Pu_i = np.dot(Ti_vec, v_a)
    Pind_i = Ti_mag * v_ind
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5
    Pi = Pu_i + Pind_i + Pp_i

    return {
        # "edge_in": edge_in_key,
        # "edge_out": edge_out_key,
        "total_power_Pi": Pi,
        "Tmaneuver": Ti_mag,
        # "acceleration": a_i
    }

# calculate the coeffiction

In [13]:
space_graph = generate_moore_neighbor_dict(grid_5x5x5,speed_map,wind_5x5x5)

print(f"\nDirected Edge Dictionary: {len(space_graph)}")


Directed Edge Dictionary: 1292


In [14]:
space_graph

{'0_0_0_0_1_0_0': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': np.float64(1.5),
  'v_level': 0,
  'v': 10.0,
  'v_vector': array([ 0., 10.,  0.]),
  'wind_vector': array([-0.70598215,  0.11661208,  3.7586217 ], dtype=float32),
  'Pi': np.float64(289.9691523192064),
  'Tmaneuver': np.float64(36.78449430524022),
  'Energy': np.float64(43.49537284788096)},
 '0_0_0_0_1_0_1': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': np.float64(1.5),
  'v_level': 1,
  'v': 15.0,
  'v_vector': array([ 0., 15.,  0.]),
  'wind_vector': array([-0.70598215,  0.11661208,  3.7586217 ], dtype=float32),
  'Pi': np.float64(632.0916696806805),
  'Tmaneuver': np.float64(45.654820457328135),
  'Energy': np.float64(63.209166968068054)},
 '0_0_4_0_1_4_0': {'startpoint': [0, 0, 4],
  'endpoint': [0, 1, 4],
  'euclidean_distance': np.float64(1.5),
  'v_level': 0,
  'v': 10.0,
  'v_vector': array([ 0., 10.,  0.]),
  'wind_vector': array([0.1703991 , 0.01085152, 2

In [ ]:
def compute_lambda(x, y, z):
    term1 = 28 + 22 * (x + y + z - 6)
    
    term2 = 17 * (
        (x - 2) * (y - 2) + 
        (x - 2) * (z - 2) + 
        (y - 2) * (z - 2)
    )
    
    term3 = 13 * (x - 2) * (y - 2) * (z - 2)
    
    lambda_val = term1 + term2 + term3
    return lambda_val

# Example usage:
result = compute_lambda(target_shape[0], target_shape[1], target_shape[2])
print(f"Recheck number of Directed Edge: {result*2*len(speed_map)}")


Recheck number of Directed Edge: 4144


In [ ]:

# Drone flies from (1,1,1) -> (2,2,2) with Level 1, then continues to (3,2,2) with Level 0
# Randomly select two consecutive edges: endpoint(edge1) == startpoint(edge2)
start_map = {}
for k, d in space_graph.items():
    start_map.setdefault(tuple(d["startpoint"]), []).append(k)

candidate_edge1 = [
    k for k, d in space_graph.items()
    if tuple(d["endpoint"]) in start_map
]

if not candidate_edge1:
    raise ValueError("No consecutive edge pair found in space_graph.")

edge1 = np.random.choice(candidate_edge1)
edge2 = np.random.choice(start_map[tuple(space_graph[edge1]["endpoint"])])
wind_vec = np.array([1.0, 0, 0]) # Wind blowing along X axis

edge1_data = space_graph[edge1]
edge2_data = space_graph[edge2]
result = calculate_energy_transition(edge1_data, edge2_data, wind_vec, drone_params, dt) # Transition time
print(f"\nEnergy transition from {edge1} to {edge2}: ")
print(f"{result['total_power_Pi']:.2f} J, Thrust force={result['Tmaneuver']:.3f} N")


Energy transition from 3_4_3_4_3_2_1 to 4_3_2_4_2_3_0: 
1592.50 J, Thrust force=123.238 N


In [17]:
space_graph

{'0_0_0_0_1_0_0': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': np.float64(1.5),
  'v_level': 0,
  'v': 10.0,
  'v_vector': array([ 0., 10.,  0.]),
  'wind_vector': array([-0.70598215,  0.11661208,  3.7586217 ], dtype=float32),
  'Pi': np.float64(289.9691523192064),
  'Tmaneuver': np.float64(36.78449430524022),
  'Energy': np.float64(43.49537284788096)},
 '0_0_0_0_1_0_1': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': np.float64(1.5),
  'v_level': 1,
  'v': 15.0,
  'v_vector': array([ 0., 15.,  0.]),
  'wind_vector': array([-0.70598215,  0.11661208,  3.7586217 ], dtype=float32),
  'Pi': np.float64(632.0916696806805),
  'Tmaneuver': np.float64(45.654820457328135),
  'Energy': np.float64(63.209166968068054)},
 '0_0_4_0_1_4_0': {'startpoint': [0, 0, 4],
  'endpoint': [0, 1, 4],
  'euclidean_distance': np.float64(1.5),
  'v_level': 0,
  'v': 10.0,
  'v_vector': array([ 0., 10.,  0.]),
  'wind_vector': array([0.1703991 , 0.01085152, 2

In [18]:
# Filter all edges starting from point (0, 0, 0)
edges_from_origin = {key: value for key, value in space_graph.items() if value['startpoint'] == start_point}

print(f"Number of edges starting from (0,0,0): {len(edges_from_origin)}")
print("\nEdges from origin:")
for key, data in edges_from_origin.items():
    astart = data['v_vector']/dt_takeoff
    Etakeoff = calculate_drone_power(data['v_vector']/2, [1,1,2], astart, drone_params)["Pi"]*dt_takeoff
    space_graph[key]["Etakeoff"] = Etakeoff  # Add takeoff energy to the edge
    print(f"{key}:   Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Etakeoff={Etakeoff:.2f} J")

Number of edges starting from (0,0,0): 6

Edges from origin:
0_2_0_0_1_0_0:   Energy=39.48, Thrust force=34.907 N, Etakeoff=671.03 J
0_2_0_0_1_0_1:   Energy=63.96, Thrust force=44.912 N, Etakeoff=1197.41 J
0_2_0_0_3_0_0:   Energy=35.73, Thrust force=33.852 N, Etakeoff=583.27 J
0_2_0_0_3_0_1:   Energy=58.56, Thrust force=43.138 N, Etakeoff=1027.02 J
0_2_0_0_3_1_0:   Energy=180.34, Thrust force=41.805 N, Etakeoff=988.87 J
0_2_0_0_3_1_1:   Energy=212.79, Thrust force=51.380 N, Etakeoff=1573.68 J


In [19]:
# Filter all edges ending at point (4, 4, 5)
edges_to_destination = {key: value for key, value in space_graph.items() if value['endpoint'] == end_point}

print(f"Number of edges ending at (4,4,5): {len(edges_to_destination)}")
print("\nEdges to destination:")
for key, data in edges_to_destination.items():
    astart = data['v_vector']/dt_landing
    Elanding = calculate_drone_power(data['v_vector']/2, [1,1,2], astart, drone_params)["Pi"]*dt_landing
    space_graph[key]["Elanding"] = Elanding  # Add landing energy to the edge
    print(f"{key}: {data['endpoint']}  Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Elanding={Elanding:.2f} J")

Number of edges ending at (4,4,5): 18

Edges to destination:
3_2_3_4_2_4_0: [4, 2, 4]  Energy=225.40, Thrust force=41.655 N, Elanding=887.58 J
3_2_3_4_2_4_1: [4, 2, 4]  Energy=279.97, Thrust force=51.598 N, Elanding=1431.56 J
3_2_4_4_2_4_0: [4, 2, 4]  Energy=102.33, Thrust force=34.648 N, Elanding=583.27 J
3_2_4_4_2_4_1: [4, 2, 4]  Energy=165.59, Thrust force=44.357 N, Elanding=1027.02 J
3_3_3_4_2_4_0: [4, 2, 4]  Energy=224.69, Thrust force=41.236 N, Elanding=889.74 J
3_3_3_4_2_4_1: [4, 2, 4]  Energy=281.29, Thrust force=50.958 N, Elanding=1441.16 J
3_3_4_4_2_4_0: [4, 2, 4]  Energy=106.07, Thrust force=34.111 N, Elanding=599.84 J
3_3_4_4_2_4_1: [4, 2, 4]  Energy=176.31, Thrust force=44.050 N, Elanding=1060.20 J
4_1_3_4_2_4_0: [4, 2, 4]  Energy=187.79, Thrust force=42.298 N, Elanding=988.87 J
4_1_3_4_2_4_1: [4, 2, 4]  Energy=223.25, Thrust force=52.468 N, Elanding=1573.68 J
4_1_4_4_2_4_0: [4, 2, 4]  Energy=38.19, Thrust force=34.536 N, Elanding=583.27 J
4_1_4_4_2_4_1: [4, 2, 4]  Energy=

In [ ]:
# Draw all edges in space_graph
fig = go.Figure()

# (Optional) draw obstacles for easier visualization
occ_idx = np.argwhere(grid_5x5x5 > 0)
for xi, yi, zi in occ_idx:
    add_cube(fig, xi, yi, zi, size=1.0, color="crimson", opacity=0.12)

# Group edges by v_level for coloring
level_colors = {0: "royalblue", 1: "orange", 2: "green"}
levels = sorted({data["v_level"] for data in space_graph.values()})

for lv in levels:
    xs, ys, zs = [], [], []
    for data in space_graph.values():
        if data["v_level"] != lv:
            continue
        x1, y1, z1 = data["startpoint"]
        x2, y2, z2 = data["endpoint"]

        # Place at cell center
        xs += [x1 + 0.5, x2 + 0.5, None]
        ys += [y1 + 0.5, y2 + 0.5, None]
        zs += [z1 + 0.5, z2 + 0.5, None]

    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(width=3, color=level_colors.get(lv, "black")),
        name=f"v_level={lv}"
    ))

# Draw start/end points
fig.add_trace(go.Scatter3d(
    x=[start_point[0] + 0.5], y=[start_point[1] + 0.5], z=[start_point[2] + 0.5],
    mode="markers", marker=dict(size=7, color="lime"), name="start"
))
fig.add_trace(go.Scatter3d(
    x=[end_point[0] + 0.5], y=[end_point[1] + 0.5], z=[end_point[2] + 0.5],
    mode="markers", marker=dict(size=7, color="red"), name="end"
))

nx, ny, nz = grid_5x5x5.shape
fig.update_layout(
    title=f"All edges in space (N={len(space_graph)})",
    scene=dict(
        xaxis=dict(title="X", range=[0, nx], dtick=1),
        yaxis=dict(title="Y", range=[0, ny], dtick=1),
        zaxis=dict(title="Z", range=[0, nz], dtick=1),
        aspectmode="cube"
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [ ]:
# Draw all edges in space_graph
fig = go.Figure()
# Select one random point (free cell)
free_cells = np.argwhere(grid_5x5x5 <= 0)
selected_point = free_cells[np.random.choice(len(free_cells))].tolist()
px, py, pz = selected_point

# Get outgoing/incoming edges of the selected point
edges_out = {k: v for k, v in space_graph.items() if v["startpoint"] == selected_point}
edges_in  = {k: v for k, v in space_graph.items() if v["endpoint"] == selected_point}

# Aggregate energies for blue -> red color scaling
local_edges = list(edges_out.values()) + list(edges_in.values())
energies = [d.get("Energy", 0.0) for d in local_edges]
emin, emax = (min(energies), max(energies)) if energies else (0.0, 1.0)

def edge_color(val, vmin, vmax):
    if vmax <= vmin:
        t = 0.5
    else:
        t = (val - vmin) / (vmax - vmin)
    r = int(255 * t)
    b = int(255 * (1 - t))
    return f"rgb({r},0,{b})"  # blue -> red

fig = go.Figure()

# Draw obstacles + legend group for show/hide
start_trace_idx = len(fig.data)
occ_idx = np.argwhere(grid_5x5x5 > 0)
for xi, yi, zi in occ_idx:
    add_cube(fig, xi, yi, zi, size=1.0, color="crimson", opacity=0.10)

for i in range(start_trace_idx, len(fig.data)):
    fig.data[i].legendgroup = "obstacle"
    fig.data[i].name = "obstacle"
    fig.data[i].showlegend = (i == start_trace_idx)

# Draw outgoing edges (same color scale)
first_out = True
for d in edges_out.values():
    x1, y1, z1 = d["startpoint"]
    x2, y2, z2 = d["endpoint"]
    c = edge_color(d.get("Energy", 0.0), emin, emax)
    fig.add_trace(go.Scatter3d(
        x=[x1 + 0.5, x2 + 0.5],
        y=[y1 + 0.5, y2 + 0.5],
        z=[z1 + 0.5, z2 + 0.5],
        mode="lines",
        line=dict(color=c, width=6),
        name="edge out",
        legendgroup="edge_out",
        showlegend=first_out
    ))
    first_out = False

# Draw incoming edges (same color scale)
first_in = True
for d in edges_in.values():
    x1, y1, z1 = d["startpoint"]
    x2, y2, z2 = d["endpoint"]
    c = edge_color(d.get("Energy", 0.0), emin, emax)
    fig.add_trace(go.Scatter3d(
        x=[x1 + 0.5, x2 + 0.5],
        y=[y1 + 0.5, y2 + 0.5],
        z=[z1 + 0.5, z2 + 0.5],
        mode="lines",
        line=dict(color=c, width=6, dash="dot"),
        name="edge in",
        legendgroup="edge_in",
        showlegend=first_in
    ))
    first_in = False

# Selected point
fig.add_trace(go.Scatter3d(
    x=[px + 0.5], y=[py + 0.5], z=[pz + 0.5],
    mode="markers",
    marker=dict(size=9, color="deepskyblue"),
    name=f"selected {selected_point}",
    legendgroup="selected",
    showlegend=True
))

# Get surrounding vertices (3x3x3), excluding obstacle vertices
nx, ny, nz = grid_5x5x5.shape
neighbor_pts = []
for dx, dy, dz in itertools.product([-1, 0, 1], repeat=3):
    xx, yy, zz = px + dx, py + dy, pz + dz
    if 0 <= xx < nx and 0 <= yy < ny and 0 <= zz < nz and grid_5x5x5[xx, yy, zz] <= 0:
        neighbor_pts.append((xx, yy, zz))

if neighbor_pts:
    xv = np.array([p[0] + 0.5 for p in neighbor_pts])
    yv = np.array([p[1] + 0.5 for p in neighbor_pts])
    zv = np.array([p[2] + 0.5 for p in neighbor_pts])

    uv = np.array([wind_5x5x5[p][0] for p in neighbor_pts])
    vv = np.array([wind_5x5x5[p][1] for p in neighbor_pts])
    wv = np.array([wind_5x5x5[p][2] for p in neighbor_pts])

    fig.add_trace(go.Cone(
        x=xv, y=yv, z=zv,
        u=uv, v=vv, w=wv,
        sizemode="absolute",
        sizeref=2.0,
        anchor="tail",
        colorscale="Viridis",
        name="wind vectors",
        legendgroup="wind",
        showlegend=True
    ))

fig.update_layout(
    title=f"Edges at point {selected_point}: out={len(edges_out)}, in={len(edges_in)}",
    scene=dict(
        xaxis=dict(title="X", range=[0, nx], dtick=1),
        yaxis=dict(title="Y", range=[0, ny], dtick=1),
        zaxis=dict(title="Z", range=[0, nz], dtick=1),
        aspectmode="cube"
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.update_traces(showscale=False, selector=dict(type="cone"))

fig.show()
# fig.show()

#  Eliminate variable

In [21]:
# Eliminate edges that end at start_point or start at end_point
edges_to_start = {
    key: data
    for key, data in space_graph.items()
    if data["endpoint"] == start_point
}

edges_from_end = {
    key: data
    for key, data in space_graph.items()
    if data["startpoint"] == end_point
}

print(edges_to_start)
print(edges_from_end)

{'0_1_0_0_2_0_0': {'startpoint': [0, 1, 0], 'endpoint': [0, 2, 0], 'euclidean_distance': np.float64(1.5), 'v_level': 0, 'v': 10.0, 'v_vector': array([ 0., 10.,  0.]), 'wind_vector': array([-0.3806429 ,  0.19799903,  5.262494  ], dtype=float32), 'Pi': np.float64(249.57641872624072), 'Tmaneuver': np.float64(34.57938942531204), 'Energy': np.float64(37.436462808936106)}, '0_1_0_0_2_0_1': {'startpoint': [0, 1, 0], 'endpoint': [0, 2, 0], 'euclidean_distance': np.float64(1.5), 'v_level': 1, 'v': 15.0, 'v_vector': array([ 0., 15.,  0.]), 'wind_vector': array([-0.3806429 ,  0.19799903,  5.262494  ], dtype=float32), 'Pi': np.float64(593.991957957309), 'Tmaneuver': np.float64(43.693973824318704), 'Energy': np.float64(59.3991957957309)}, '0_3_0_0_2_0_0': {'startpoint': [0, 3, 0], 'endpoint': [0, 2, 0], 'euclidean_distance': np.float64(1.5), 'v_level': 0, 'v': 10.0, 'v_vector': array([  0., -10.,   0.]), 'wind_vector': array([-0.32605803,  0.22433166,  5.7417526 ], dtype=float32), 'Pi': np.float64(

In [22]:
# Eliminate edge in obstacle

In [23]:
# Eliminate edge with thrust force above a certain threshold (e.g., 15 N)

In [ ]:
# Remove edges in space_graph whose key is in edges_to_start or edges_from_end
keys_to_remove = set(edges_to_start.keys()) | set(edges_from_end.keys())

removed = 0
for k in keys_to_remove:
    if k in space_graph:
        del space_graph[k]
        removed += 1

print(f"Deleted {removed} keys from space_graph")
print(f"Remaining edges: {len(space_graph)}")

Đã xoá 24 key khỏi space_graph
Số cạnh còn lại: 1268


# QUBO matrix creation

In [25]:
# Create QUBO matrix from space_graph
edge_keys = list(space_graph.keys())
N = len(edge_keys)
edge_index = {key: idx for idx, key in enumerate(edge_keys)}

print(f"Number of edges (N): {N}")
print(f"QUBO matrix size: {N} x {N}")

# Initialize QUBO matrix
Q = np.zeros((N, N), dtype=float)


Number of edges (N): 1268
QUBO matrix size: 1268 x 1268


## Objective function

In [ ]:
# Assign diagonal values Q[i,i] = 3 for edges with the startpoint
list_edges_start = {
    key: space_graph[key]
    for key in edge_index
    if space_graph[key]["startpoint"] == start_point
}
max_energy_start = 0
for key in list_edges_start:
    idx = edge_index[key]  # get key index in graph
    bufVal= 0.5*space_graph[key]["Energy"] + space_graph[key].get("Etakeoff", 0.0)
    Q[idx, idx] += bufVal
    if bufVal > max_energy_start:
        max_energy_start = bufVal

print(f"Max energy for edges from start point: {max_energy_start:.2f} J")


Max energy for edges from start point: 1680.07 J


In [ ]:
# Assign diagonal values Q[i,i] = 3 for edges with the endpoint
list_edges_end = {
    key: space_graph[key]
    for key in edge_index
    if space_graph[key]["endpoint"] == end_point
}
max_energy_end = 0

for key in list_edges_end:
    idx = edge_index[key]  # get key index in graph
    bufVal = 0.5*space_graph[key]["Energy"] + space_graph[key].get("Elanding", 0.0)
    Q[idx, idx] += bufVal
    if bufVal > max_energy_end:
        max_energy_end = bufVal
print(f"Max energy for edges to end point: {max_energy_end:.2f} J")

Max energy for edges to end point: 1734.97 J


In [ ]:
# Loop through all points in space, excluding start and end
start_t = tuple(start_point)
end_t = tuple(end_point)

all_mid_points = []
for x, y, z in itertools.product(range(nx), range(ny), range(nz)):
    p = (x, y, z)
    if p == start_t or p == end_t:
        continue
    all_mid_points.append(p)



In [ ]:
max_pair_energy = 0
for  target_xyz in all_mid_points:
    prefix = f"{target_xyz[0]}_{target_xyz[1]}_{target_xyz[2]}_"
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["startpoint"]) == target_xyz
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["endpoint"]) == target_xyz
    }
    # print(f"Number of keys starting with {target_xyz}: {len(edges_prefix)}")
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]
            change_dir = calculate_energy_transition(space_graph[edges_in], space_graph[edges_out], wind_vec, drone_params, dt)
            bufVal = 0.5*space_graph[edges_in]["Energy"] + 0.5*space_graph[edges_out]["Energy"] + change_dir["total_power_Pi"]
            Q[idx_in, idx_out] += bufVal
            if bufVal > max_pair_energy:
                max_pair_energy = bufVal

max_pair_energy

np.float64(2667.908885424414)

## Constraint

In [ ]:
for key in list_edges_start:
    idx = edge_index[key]  # get key index in graph
    Q[idx, idx] -= max_energy_start

for i in range(len(list_edges_start)):
    for j in range(i+1, len(list_edges_start)):
        idx_i = edge_index[list(list_edges_start.keys())[i]]
        idx_j = edge_index[list(list_edges_start.keys())[j]]

        Q[idx_i][idx_j] += 2*max_energy_start

In [ ]:
for key in list_edges_end:
    idx = edge_index[key]  # get key index in graph
    Q[idx, idx] -= max_energy_end

for i in range(len(list_edges_end)):
    for j in range(i+1, len(list_edges_end)):
        idx_i = edge_index[list(list_edges_end.keys())[i]]
        idx_j = edge_index[list(list_edges_end.keys())[j]]

        Q[idx_i][idx_j] += 2*max_energy_end

In [32]:
max_pair_energy = 0
for  target_xyz in all_mid_points:
    prefix = f"{target_xyz[0]}_{target_xyz[1]}_{target_xyz[2]}_"
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["startpoint"]) == target_xyz
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["endpoint"]) == target_xyz
    }
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        Q[idx_in, idx_in] += max_pair_energy

    for edges_out in edges_out_list:
        idx_out = edge_index[edges_out]
        Q[idx_out, idx_out] += max_pair_energy

    for i in range(len(edges_in_list)):
        for j in range(i+1, len(edges_in_list)):
            idx_i = edge_index[list(edges_in_list.keys())[i]]
            idx_j = edge_index[list(edges_in_list.keys())[j]]

            Q[idx_i][idx_j] += 2*max_pair_energy

    for i in range(len(edges_out_list)):
        for j in range(i+1, len(edges_out_list)):
            idx_i = edge_index[list(edges_out_list.keys())[i]]
            idx_j = edge_index[list(edges_out_list.keys())[j]]

            Q[idx_i][idx_j] += 2*max_pair_energy

    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]
            Q[idx_in, idx_out] +=-2*max_pair_energy

In [33]:
for key in edge_index:
    if space_graph[key]["Tmaneuver"] > Tmax:  # Threshold for high thrust
        idx = edge_index[key]
        Q[idx, idx] += max_pair_energy

In [ ]:
max_pair_energy = 0
for  target_xyz in all_mid_points:
    prefix = f"{target_xyz[0]}_{target_xyz[1]}_{target_xyz[2]}_"
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["startpoint"]) == target_xyz
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["endpoint"]) == target_xyz
    }
    # print(f"Number of keys starting with {target_xyz}: {len(edges_prefix)}")
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]
            change_dir = calculate_energy_transition(space_graph[edges_in], space_graph[edges_out], wind_vec, drone_params, dt)
            if change_dir["Tmaneuver"] > Tmax:
                Q[idx_in, idx_out] += max_pair_energy

In [35]:
Q

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(1268, 1268))